# Essential App Functions 
This notebook implements the functions needed for our Emotion Recognition App.

## Imports

In [ ]:
#!pip install PyPDF2 python-docx pillow opencv-python librosa SpeechRecognition moviepy pydub
#!pip install git+https://github.com/openai/whisper.git


import os
import tempfile
import PyPDF2
import cv2
import librosa
#import speech_recognition as sr
from docx import Document
from PIL import Image
from moviepy.editor import VideoFileClip
import numpy as np
import whisper


## File format identification

In [ ]:

class FileProcessor:
    def __init__(self):
        self.file_types = {
            'text': ['txt', 'pdf', 'docx'],
            'audio': ['mp3', 'wav', 'sph'],
            'image': ['jpg', 'jpeg', 'png'],
            'video': ['mp4', 'avi', 'mov']
        }
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
        self.recognizer = whisper.load_model("base") #sr.Recognizer()

    def process_file(self, file_path):
        """Main processing method"""
        if not os.path.isfile(file_path):
            return {"error": "File not found"}
            
        file_type = self._detect_file_type(file_path)
        
        try:
            if file_type == 'text':
                return self._process_text(file_path)
            elif file_type == 'audio':
                return self._process_audio(file_path)
            elif file_type == 'image':
                return self._process_image(file_path)
            elif file_type == 'video':
                return self._process_video(file_path)
            else:
                return {"error": "Unsupported file type"}
        except Exception as e:
            return {"error": str(e)}

    def _detect_file_type(self, file_path):
        """Detect file type category"""
        ext = file_path.split('.')[-1].lower()
        for category, extensions in self.file_types.items():
            if ext in extensions:
                return category
        return 'unknown'

    def _process_text(self, file_path):
        """Process text-based files"""
        ext = file_path.split('.')[-1].lower()
        text = ''
        
        if ext == 'txt':
            with open(file_path, 'r') as f:
                text = f.read()
        elif ext == 'pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                text = '\n'.join([page.extract_text() for page in reader.pages])
        elif ext == 'docx':
            doc = Document(file_path)
            text = '\n'.join([para.text for para in doc.paragraphs])
            
        return {"text": text}

    def _process_audio(self, file_path):
        """Process audio files with speech recognition"""
        # Load audio with librosa
        audio, sr = librosa.load(file_path, sr=16000)
        
        # Extract transcript using Whisper
        transcript = self.recognizer.transcribe(audio)
            
        return {
            "text": transcript,
            "audio": {
                "raw": audio,
                "sample_rate": sr
            }
        }

    def _process_image(self, file_path):
        """Process image files with face detection"""
        # Load image and convert to grayscale
        img = cv2.imread(file_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Detect faces
        faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)
        
        if len(faces) == 0:
            return {"error": "No faces detected"}
            
        # Get first face and convert to PIL Image
        x, y, w, h = faces[0]
        face_img = img[y:y+h, x:x+w]
        pil_image = Image.fromarray(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
        
        return {"image": pil_image}

    def _process_video(self, file_path):
        """Process video files with frame extraction and audio processing"""
        result = {}
        
        # Extract audio
        with tempfile.NamedTemporaryFile(suffix='.wav') as tmpfile:
            video = VideoFileClip(file_path)
            video.audio.write_audiofile(tmpfile.name)
            audio_result = self._process_audio(tmpfile.name)
            result.update(audio_result)
        
        # Extract frames with faces
        cap = cv2.VideoCapture(file_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_interval = int(fps * 1)  # Every 5 seconds
        frame_count = 0
        frames = []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            if frame_count % frame_interval == 0:
                # Detect faces
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = self.face_cascade.detectMultiScale(gray, 1.3, 5)
                
                if len(faces) > 0:
                    x, y, w, h = faces[0]
                    face_img = frame[y:y+h, x:x+w]
                    pil_image = Image.fromarray(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
                    frames.append(pil_image)
                    
            frame_count += 1
            
        cap.release()
        result["frames"] = frames
        
        return result



  Cloning https://github.com/openai/whisper.git to /private/var/folders/0s/lm29041s6hv5pq_lc3jblq540000gn/T/pip-req-build-r08xzxh0
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /private/var/folders/0s/lm29041s6hv5pq_lc3jblq540000gn/T/pip-req-build-r08xzxh0
  Resolved https://github.com/openai/whisper.git to commit 517a43ecd132a2089d85f4ebc044728a71d49f6e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Usage Example
processor = FileProcessor()

# Process text file
text_result = processor.process_file("/Users/sofiafernandes/Documents/Repos/MEIM-ano1-sem2/TFM-SC/lumios.pdf")
print(text_result)



{'text': 'LumiosConnectBetter,LiveBetter\nTeam·GiovanniRobertoFranzan:FormerGoogler,visionar ywithexpertiseinComput erVisionandGaming.·Giandomenic oIannetti:ProfessorofNeuroscienceatUniversityCollegeLondon\nProblem"OurEpidemicofLonelinessandIsolation,"highligh tedbyU.S.SurgeonGeneralDr.VivekMurthy,isacriticalpublichealthissue.Lonelinessandsocialanxietyaffectmillions,leadingtoseverementalandphysicalhealthproblems,includingdepressionandheartdisease.Thisdisconnectionhindersrelationship-building ,especiallyindatingandprofessionalsettings.\nDespitemanyonlineplatformsteachinghardskills,there\'sasignificantgapinresourcesfordevelopingcrucialsoftskills.Addressingthisgapisessentialforimprovingwell-beingandfosteringstrongercommunities.\nSolutionWeaddressthisproblemwithanAI-poweredplatformofferingreal-timeguidancethroughvideoandaudioanalysistoimproveconversationalskillswithpersonalizedfeedback.Byincorporatingexposuretherapy,wehelpusersgraduallyfacesocialfears,fromcasualconversationstoprofessionals

In [3]:
processor = FileProcessor()
# Process text file
text_result = processor.process_file("/Users/sofiafernandes/Documents/Repos/MEIM-ano1-sem2/TFM-SC/notebooks/list.docx")
print(text_result)

{'text': '\nList of bigger supermarkets in Nigeria, Lagos state.\n\n1 Shoprite: Considered the largest supermarket chain in Lagos, with multiple locations across the city. It offers a wide range of products including groceries, beauty items, electronics and more.\n\n2 Spar: Another major supermarket chain with over 8 branches in Lagos. Spar provides a variety of groceries, household goods, electronics and home decor.\n\n3 Prince Ebeano Supermarket: One of the biggest retail supermarkets in Lagos, with 5 locations in Lagos and Abuja. Known for its wide selection of quality products at discounted prices.\n\n4 Hubmart: A relatively new supermarket chain that has gained popularity in Lagos. Hubmart offers fresh foods, seafood, home appliances and good customer service.\n\n 5 Justrite: A multi-location supermarket chain in Lagos that serves as a one-stop grocery store. Justrite has a loyalty program and clearly priced products on neatly arranged shelves.\n\n   This are the major 5s that wer

In [4]:
processor = FileProcessor()
# Process audio file
audio_result = processor.process_file("/Users/sofiafernandes/Documents/Repos/MEIM-ano1-sem2/TFM-SC/test.wav")
print(audio_result)

{'text': {'text': " Excuse me. Do you have your forms? Yeah. Can you see them? Is there a problem? Who told you to get in this line? You did. No. You were standing at the beginning and you direct me. Okay, but I didn't tell you to get in this line if you're filling out this particular form. Well, what is this for? What is it? Zx4. You can't... This is not the line for the Zx4. If you're going to fill out the Zx4, you need to have a different form of ID. What? I'm getting an ID. This is why I'm here. No, I need another set of ID to prove that this is actually you. How am I supposed to get an ID without an ID? How does the person get an ID in the first place? I don't know, but I need an ID to pass this form along. I can't just send it along without an ID. You're here to get an ID. No, I need another ID. A separate one. Like what? Like a birth certificate? A birth certificate. How have a birth certificate? A student ID. Did you go to school? Anything? Yes, but my wallet was stolen. I don'

In [6]:
processor = FileProcessor()
# Process image
image_result = processor.process_file("/Users/sofiafernandes/Documents/Repos/MEIM-ano1-sem2/TFM-SC/notebooks/imgs/sad.jpg")
if "image" in image_result:
    image_result["image"].show()

In [7]:
processor = FileProcessor()
# Process video
video_result = processor.process_file("/Users/sofiafernandes/Documents/Repos/MEIM-ano1-sem2/TFM-SC/01-01-05-02-02-01-01.mp4")
#print(f"Video result contains {len(video_result['frames'])} frames")
print(video_result)

MoviePy - Writing audio in /var/folders/0s/lm29041s6hv5pq_lc3jblq540000gn/T/tmpx3_nwoau.wav


MoviePy - Done.


{'text': {'text': ' Dogs are sitting by the door!', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 4.0, 'text': ' Dogs are sitting by the door!', 'tokens': [50364, 35504, 366, 3798, 538, 264, 2853, 0, 50564], 'temperature': 0.0, 'avg_logprob': -0.46198205947875975, 'compression_ratio': 0.7837837837837838, 'no_speech_prob': 0.08906954526901245}], 'language': 'en'}, 'audio': {'raw': array([-1.9290547e-11,  5.7371101e-11,  5.8553030e-11, ...,
       -4.3314006e-04, -1.3577106e-04,  1.0749898e-03], dtype=float32), 'sample_rate': 16000}, 'frames': [<PIL.Image.Image image mode=RGB size=417x417 at 0x7F899B8E14C0>, <PIL.Image.Image image mode=RGB size=401x401 at 0x7F898006DF70>, <PIL.Image.Image image mode=RGB size=397x397 at 0x7F89574EFBB0>, <PIL.Image.Image image mode=RGB size=402x402 at 0x7F897FF3C8E0>, <PIL.Image.Image image mode=RGB size=397x397 at 0x7F897FF3C280>]}


## Emotion Recognizer

In [27]:
#!pip install litellm 
from litellm import completion
from deepface import DeepFace
import opensmile
import cv2
import pickle
import librosa
import numpy as np
import joblib


class TextEmotionRecognizer:
    def __init__(self, llm_model="phi4-mini"):
        self.llm_model = llm_model
        
    
    def analyze(self, text):
        response = completion(
            model=f"ollama_chat/{self.llm_model}", 
            messages=[{ 
                "content": f"Respond with only one word (lower case and no extra characters) from these emotions ['sad', 'happy', 'disgusted', 'surprised'] respecting to the most expressed emotion in the following piece of text: {text}",
                "role": "user"}],    
        )
        print(response.choices[0].message.content)
        return response.choices[0].message.content.strip()
    
    
class SpeechEmotionRecognizer:
    def __init__(self, model_path='svm_model.pkl'):

        # Load the saved model from the file
        with open(model_path, 'rb') as file:
            self.model = pickle.load(file)

        #self.scaler = joblib.load('models/feature_scaler.pkl')

    def extract_features(self, audio, sr):
        # Extract features using OpenSMILE
        smile = opensmile.Smile(
            feature_set=opensmile.FeatureSet.GeMAPSv01b,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
        features = smile.process_signal(audio, sr)
        return features.values #.flatten().reshape(1, -1)

    def extract_features_old(self, audio, sr):
        # Implement GeMAPS or eGeMAPS feature extraction
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        features = np.concatenate([
            np.mean(mfcc, axis=1),
            np.std(mfcc, axis=1)
        ])
        return self.scaler.transform([features])
    
    def analyze(self, audio, sr):
        features = self.extract_features(audio, sr)
        prediction = self.model.predict(features)
        return {
            'emotions': prediction, #dict(zip(self.model.classes_, prediction)),
            #'dominant_emotion': self.model.classes_[np.argmax(prediction)]
        }
    
class FaceEmotionRecognizer:
    def __init__(self):
        #backends = ["opencv", "ssd", "mtcnn", "dlib", "retinaface"]
        self.backend = 'mtcnn'  # Use lightweight backend (SFace)
    
    def analyze_image(self, image):
        # convert PIL image to numpy array
        image_array = np.array(image)
        results = DeepFace.analyze(
            img_path=image_array,
            actions=['emotion'],
            detector_backend=self.backend,
            #enforce_detection=False
        )
        return {
            'emotions': results[0]['emotion'],
            'dominant_emotion': results[0]['dominant_emotion']
        }
    
    def analyze_video_frames(self, frames):
        return [self.analyze_image(frame) for frame in frames]

In [31]:
ter = TextEmotionRecognizer()
text = video_result["text"]["text"]
text_emotion = ter.analyze(text)
print(f"Text Emotion: {text_emotion}")
ser = SpeechEmotionRecognizer()
audio = video_result["audio"]["raw"]
sr = video_result["audio"]["sample_rate"]
audio_emotion = ser.analyze(audio, sr)
print(f"Audio Emotion: {audio_emotion}")
fer = FaceEmotionRecognizer()
frames = video_result["frames"]
face_emotion = fer.analyze_video_frames(frames)
print(f"Face Emotion: {face_emotion}")

Happy
Text Emotion: Happy
Audio Emotion: {'emotions': array(['anger'], dtype=object)}
1/1 [==============================] - 0s 29ms/step
Face Emotion: [{'emotions': {'angry': 12.861937241323323, 'disgust': 4.808646612328695, 'fear': 8.707469408813786, 'happy': 0.13154256917773852, 'sad': 73.13641629977032, 'surprise': 0.005827268283799905, 'neutral': 0.3481612358571606}, 'dominant_emotion': 'sad'}, {'emotions': {'angry': 56.86090588569641, 'disgust': 2.3301975801587105, 'fear': 4.089658334851265, 'happy': 0.2290672855451703, 'sad': 22.41366356611252, 'surprise': 5.997591093182564, 'neutral': 8.078914135694504}, 'dominant_emotion': 'angry'}, {'emotions': {'angry': 25.024351474720063, 'disgust': 65.39717729125293, 'fear': 1.8159174989560247, 'happy': 0.029367033267412093, 'sad': 3.7892012996433615, 'surprise': 0.28225131227805467, 'neutral': 3.6617293314064345}, 'dominant_emotion': 'disgust'}, {'emotions': {'angry': 99.48180910836513, 'disgust': 0.00437104380258527, 'fear': 0.2788256554

## Emotion Recognition Assistant

In [ ]:
class Chatbot:
    def __init__(self):
        self.llm_model = "phi4-mini"
        self.chatbot = completion(
            model=f"ollama_chat/{self.llm_model}",
            messages=[{
                "role": "system",
                "content": "You are a helpful assistant."
            }]
        )
        self.chatbot.start()
        self.chatbot.send_message("Hello! How can I assist you today?")
        self.chatbot.send_message("I can help you with text, audio, image, and video analysis.")
        self.chatbot.send_message("I can also analyze emotions in text, audio, and images.")
        self.chatbot.send_message("Feel free to ask me anything!")
        self.chatbot.send_message("What would you like to know?")


    def send_message(self, message):
        response = self.chatbot.send_message(message)
        return response.choices[0].message.content.strip()
    
    

class EmotionRecognitionAssistant:
    def __init__(self):
        self.file_processor = FileProcessor()
        self.text_recognizer = TextEmotionRecognizer()
        self.speech_recognizer = SpeechEmotionRecognizer()
        self.face_recognizer = FaceEmotionRecognizer()
        self.chatbot = Chatbot()

    def analyze(self, file_path):
        processor = FileProcessor()
        result = processor.process_file(file_path)
        
        if "error" in result:
            return result
        
        text = result.get("text", {}).get("text", "")
        audio = result.get("audio", {}).get("raw", None)
        sr = result.get("audio", {}).get("sample_rate", None)
        frames = result.get("frames", [])
        
        text_emotion = self.text_recognizer.analyze(text) if text else None
        audio_emotion = self.speech_recognizer.analyze(audio, sr) if audio is not None else None
        face_emotion = self.face_recognizer.analyze_video_frames(frames) if frames else None
        
        return {
            "text_emotion": text_emotion,
            "audio_emotion": audio_emotion,
            "face_emotion": face_emotion
        }

Audio Emotion: {'emotions': array(['anger'], dtype=object)}


1/1 [==============================] - 0s 29ms/step
Face Emotion: [{'emotions': {'angry': 12.861937241323323, 'disgust': 4.808646612328695, 'fear': 8.707469408813786, 'happy': 0.13154256917773852, 'sad': 73.13641629977032, 'surprise': 0.005827268283799905, 'neutral': 0.3481612358571606}, 'dominant_emotion': 'sad'}, {'emotions': {'angry': 56.86090588569641, 'disgust': 2.3301975801587105, 'fear': 4.089658334851265, 'happy': 0.2290672855451703, 'sad': 22.41366356611252, 'surprise': 5.997591093182564, 'neutral': 8.078914135694504}, 'dominant_emotion': 'angry'}, {'emotions': {'angry': 25.024351474720063, 'disgust': 65.39717729125293, 'fear': 1.8159174989560247, 'happy': 0.029367033267412093, 'sad': 3.7892012996433615, 'surprise': 0.28225131227805467, 'neutral': 3.6617293314064345}, 'dominant_emotion': 'disgust'}, {'emotions': {'angry': 99.48180910836513, 'disgust': 0.00437104380258527, 'fear': 0.27882565543506477, 'happy': 0.00024830741610737714, 'sad': 0.12930027048589413, 'surprise': 0.04

In [ ]:
import os
from typing import List, Dict
from PIL.Image import Image as PILImage
import numpy as np
import librosa
import joblib  # For loading trained audio classifier
from deepface import DeepFace
from smolagents import CodeAgent, LiteLLMModel  # Correct import

class EmotionRecognitionAssistant:
    def __init__(self, llm_api_base="http://localhost:11434", llm_api_key="YOUR_API_KEY", llm_model_id="ollama_chat/llama2", audio_classifier_path=None):
        self.llm_model = LiteLLMModel(
            model_id=llm_model_id,
            api_base=llm_api_base,
            api_key=llm_api_key,
            num_ctx=8192,
        )
        self.face_emotion_model = "mtcnn" # Or another lightweight DeepFace model
        self.audio_feature_extractor = self._load_audio_feature_extractor()
        self.audio_emotion_classifier = self._load_audio_emotion_classifier(audio_classifier_path)
        self.report_generator = ReportGenerator(self.llm_model) # Instantiate ReportGenerator

    def _load_audio_feature_extractor(self):
        # Placeholder: Implement loading your GeMAPS or other feature extractor
        # For example, if you pre-calculated features, this might not be needed here.
        # If using a library, load it here.
        print("Audio feature extractor initialized (placeholder).")
        return None

    def _load_audio_emotion_classifier(self, model_path):
        if model_path and os.path.isfile(model_path):
            return joblib.load(model_path)
        else:
            print("Warning: Audio emotion classifier not loaded.")
            return None

    def recognize_emotion(self, data: Dict) -> Dict:
        """Main method to recognize emotions from processed data."""
        emotions = {}
        transcripts = {}

        if "text" in data and data["text"]:
            text_emotion = self._analyze_text_emotion(data["text"])
            emotions["text"] = text_emotion
            transcripts["full_text"] = data["text"]

        if "image" in data and isinstance(data["image"], PILImage):
            face_emotion = self._analyze_facial_emotion(data["image"])
            emotions["face"] = face_emotion

        if "frames" in data and data["frames"]:
            frame_emotions = [self._analyze_facial_emotion(frame) for frame in data["frames"]]
            # Simple aggregation: average the emotion scores across frames
            if frame_emotions:
                averaged_face_emotion = self._average_emotions(frame_emotions)
                emotions["face"] = averaged_face_emotion

        if "audio" in data and "raw" in data["audio"] and data["audio"]["raw"] is not None and self.audio_emotion_classifier:
            audio_features = self._extract_audio_features(data["audio"]["raw"], data["audio"]["sample_rate"])
            audio_emotion = self._predict_audio_emotion(audio_features)
            emotions["audio"] = audio_emotion
            if "text" in data: # Transcript was already extracted in FileProcessor
                transcripts["audio_transcript"] = data["text"]

        # Fuse emotions
        fused_emotion = self._fuse_emotions(emotions)

        # Generate report
        report = self.report_generator.generate_report(emotions, transcripts)

        return {"emotions": emotions, "fused_emotion": fused_emotion, "report": report}
    
    def generate_report(self, emotions: Dict, transcripts: Dict) -> str:
        """Generates a natural language report of the recognized emotions and transcripts."""
        report_parts = []

        if "full_text" in transcripts:
            report_parts.append(f"The full text of the document was: '{transcripts['full_text']}'.")
            if "text" in emotions and "primary" in emotions["text"]:
                report_parts.append(f"The primary emotion detected in the text was '{emotions['text']['primary']}' with the following justification: {emotions['text']['justification']}.")

        if "audio_transcript" in transcripts:
            report_parts.append(f"The transcribed speech from the audio was: '{transcripts['audio_transcript']}'.")
            if "audio" in emotions and "predicted" in emotions["audio"]:
                report_parts.append(f"The emotion predicted from the audio was '{emotions['audio']['predicted']}'.")

        if "face" in emotions and isinstance(emotions["face"], dict) and "dominant_emotion" in emotions["face"]:
            report_parts.append(f"The dominant facial emotion detected was '{emotions['face']['dominant_emotion']}' with the following emotion scores: {emotions['face']}.")
        elif "face" in emotions and isinstance(emotions["face"], dict) and "scores" in emotions["face"]: # Averaged scores
            report_parts.append(f"The average facial emotion scores across detected faces/frames were: {emotions['face']}.")

        if not report_parts:
            return "No emotions or relevant data were detected to generate a report."

        full_report_prompt = "Generate a concise natural language report summarizing the following information:\n" + "\n".join(report_parts)
        try:
            report_response = self.llm_model.predict(full_report_prompt)
            return report_response
        except Exception as e:
            return f"Error generating report: {str(e)}"

    def _analyze_text_emotion(self, text: str) -> Dict:
        """Analyze emotion in text using smolagents LLMs."""
        prompt = f"""Analyze the emotion expressed in the following text. 
        Identify the primary emotion and provide a score (0-1) for each of the following emotions: 
        joy, sadness, anger, fear, surprise, disgust, neutral. Justify your primary emotion.
        Text: "{text}"
        """
        try:
            response = self.llm_model.predict(prompt)
            # Parse the response to extract emotion scores and justification
            # This will depend on the exact output format of the LLM.
            # You might need more sophisticated parsing here.
            return {"primary": "neutral", "scores": {"joy": 0.0, "sadness": 0.0, "anger": 0.0, "fear": 0.0, "surprise": 0.0, "disgust": 0.0, "neutral": 1.0}, "justification": response}
        except Exception as e:
            print(f"Error analyzing text emotion: {e}")
            return {"error": str(e)}

    def _analyze_facial_emotion(self, image: PILImage) -> Dict:
        """Analyze emotion in a facial image using DeepFace."""
        try:
            result = DeepFace.analyze(np.array(image), actions=['emotion'], detector_backend=self.face_emotion_model, silent=True)
            if result:
                return result[0]['emotion']
            else:
                return {"error": "No face detected by DeepFace"}
        except Exception as e:
            print(f"Error analyzing facial emotion: {e}")
            return {"error": str(e)}

    def _extract_audio_features(self, audio: np.ndarray, sample_rate: int) -> np.ndarray:
        """Extract acoustic features from audio."""
        # Placeholder: Implement your GeMAPS or other feature extraction here.
        # Example using librosa for a basic feature (replace with your actual features)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        return np.mean(mfccs.T, axis=0).reshape(1, -1) # Reshape for single prediction

    def _predict_audio_emotion(self, features: np.ndarray) -> Dict:
        """Predict emotion from extracted audio features."""
        if self.audio_emotion_classifier:
            prediction = self.audio_emotion_classifier.predict(features)
            # Assuming your classifier outputs a single emotion label
            return {"predicted": prediction[0]}
        else:
            return {"error": "Audio emotion classifier not loaded"}

    def _average_emotions(self, list_of_emotion_dicts: List[Dict]) -> Dict:
        """Simple averaging of emotion scores from multiple frames."""
        if not list_of_emotion_dicts:
            return {}
        all_emotions = list_of_emotion_dicts[0].keys()
        averaged_emotions = {emotion: 0.0 for emotion in all_emotions if isinstance(list_of_emotion_dicts[0][emotion], (int, float))}
        num_samples = len(list_of_emotion_dicts)
        for emotion_dict in list_of_emotion_dicts:
            for emotion, score in emotion_dict.items():
                if emotion in averaged_emotions and isinstance(score, (int, float)):
                    averaged_emotions[emotion] += score
        return {k: v / num_samples for k, v in averaged_emotions.items()}

    def _fuse_emotions(self, emotions: Dict) -> Dict:
        """Placeholder for emotion fusion logic."""
        # Implement your emotion fusion strategy here.
        # This could involve weighted averaging based on modality reliability, etc.
        if not emotions:
            return {"fused": "neutral"}
        elif len(emotions) == 1:
            return {"fused": list(emotions.values())[0].get("primary") or list(emotions.values())[0].get("predicted") or "neutral"}
        else:
            # Simple majority voting (can be improved)
            all_predicted = []
            for modality, emotion_data in emotions.items():
                if isinstance(emotion_data, dict):
                    if "primary" in emotion_data:
                        all_predicted.append(emotion_data["primary"])
                    elif "predicted" in emotion_data:
                        all_predicted.append(emotion_data["predicted"])

            if not all_predicted:
                return {"fused": "neutral"}

            from collections import Counter
            emotion_counts = Counter(all_predicted)
            most_common = emotion_counts.most_common(1)
            return {"fused": most_common[0][0] if most_common else "neutral"}


    

In [11]:
class MultimodalFusion:
    def __init__(self, weights=None):
        self.weights = weights or {
            'text': 0.4,
            'face': 0.4,
            'audio': 0.2
        }
    
    def fuse(self, text_results, face_results, audio_results):
        # Implement late fusion strategy
        combined = {}
        
        # Normalize and weight results
        for modality, weight in self.weights.items():
            results = locals()[f"{modality}_results"]
            for emotion, score in results['emotions'].items():
                combined[emotion] = combined.get(emotion, 0) + score * weight
        
        total = sum(combined.values())
        normalized = {k: v/total for k, v in combined.items()}
        
        return {
            'combined_emotions': normalized,
            'dominant_emotion': max(normalized, key=normalized.get)
        }

## Chatbot Assistant

In [ ]:

from smolagents import LiteLLMModel

class ChatbotAssistant:
    def __init__(self, llm_model: LiteLLMModel):
        self.llm_model = llm_model
        self.knowledge = ""

    def load_report(self, report: str):
        """Loads the emotion analysis report into the chatbot's knowledge."""
        self.knowledge = f"The emotion analysis report is as follows: '{report}'. Based on this report, act as a supportive and insightful coach."

    def respond_to_user(self, user_input: str) -> str:
        """Generates a response to the user's input based on the loaded report."""
        if not self.knowledge:
            return "Please load an emotion analysis report first."

        prompt = f"""{self.knowledge}

        The user's query is: '{user_input}'. Provide a helpful and coaching-oriented response based on the information in the report.
        """

        try:
            response = self.llm_model.predict(prompt)
            return response.strip()
        except Exception as e:
            return f"Error generating chatbot response: {str(e)}"